# Experiment 3: representation, action channels, and whole-body support
Uses completed **Exp1** recordings for each checkpoint's **own-trajectory** representation quality, and **Exp2** separately for channel/support responses. Methods share experimental conditions, not realized transitions. No cross-policy replay, simulator, optimizer, training runner, or new collection. Representation cells do not require Exp2 recordings.

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/exp3_representation.py").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
import exp3_representation as analysis

## Configuration
Completed Exp1 recordings under `recording_roots` are searched recursively for all available methods. Set `tasks` to a list of task IDs to select a subset, or set `methods` to an explicit mapping for checkpoint relocation/separate checkpoint cohorts (leave `tasks=None`). Exp2 and incomplete runs are excluded.
Use trusted checkpoints only. An omitted checkpoint uses the path recorded in metadata; an explicit path must have the same SHA256. Discovery never substitutes the latest trained model. Multiple checkpoints for one method require explicit selection/separate IDs. **All eligible native transitions are used by default**; budgets/quotas are ignored unless `balanced_resampling=True`. This optional mode is deterministic balanced **subsampling without replacement**, separately per checkpoint and loading range, not bootstrap resampling. Strict protocol checks may require narrower roots.

Stored predictions are used first, independently per metric. Missing decoder fields may be filled by optional frozen inference when native inputs and matching checkpoint weights are available; stored outputs are never overwritten. Histories/checkpoints are not prerequisites for stored force metrics. Offline BARD is initialized only when requested dynamics cannot be obtained from recorded successor predictions or recorded terms. `inference_batch_size` bounds batch memory; trial/seed aggregation is unchanged.

**Runtime:** `num_workers=1` preserves serial reading; try 2-4 for compressed chunk loading. At most twice that many chunks are in flight, so watch RAM/storage throughput. Workers perform CPU decompression/schema checks only. Models and batched BARD remain single-process; `dynamics_batch_size` still controls GPU memory. `timings.json` records per-run wall time, summed worker read seconds (not wall time), fallback/dynamics time and cache hits. Numeric fallback/physics caches retain checkpoint identity, mechanics, numerical inputs and relevant code provenance; stored predictions are never replaced. Rendering changes can reuse them, but recorded-data metrics/aggregation and the separate Exp2 support analysis still run. Delete `cache_dir` to reclaim space or after external dynamics-library upgrades.

In [2]:
from b1z1_paper_names import DISPLAY_NAMES, LATEX_METHOD_NAMES
# Paper labels differ from implementation ablation numbers; task IDs stay unchanged.
CONFIG = {
    # Figure typography in points; requires installed Times New Roman.
    "figure_title_font_size": 12,
    "title_font_size": 11,
    "axis_label_font_size": 10,
    "tick_font_size": 9,
    "legend_font_size": 9,
    "annotation_font_size": 8,
    "recording_roots": [str(ROOT / "evaluation/exp1")],
    "tasks": None,  # All discovered tasks, or e.g. ["b1z1_pact", "b1z1_ppo_pos"].
    "methods": None,  # Optional explicit {ID: {"task": ..., "checkpoint": ...}}.
    "evaluation_seeds": [0, 1, 2],
    "conditions": ["nominal", "ee-disturbance", "combined-disturbance"],
    "balanced_resampling": False,  # False: ALL eligible native transitions; budgets/quotas ignored.
    "sampling_budget": None,  # Only when enabled: cap per checkpoint AND loading range.
    "samples_per_stratum": 32,  # Only when enabled and budget=None: condition/motion/phase/seed quotas.
    "inference_batch_size": 1024,  # Bounded inference batches across recording chunks.
    "dynamics_batch_size": 1024,  # Batch offline BARD; reduce if GPU memory is limited.
    "cpu_blas_threads": 1,  # Avoid thread-team overhead in small joint-space solves.
    "num_workers": 10,  # Try 2-4 CPU chunk readers; GPU inference/BARD remain in this process.
    "cache_enabled": True,  # Reuse optional decoder predictions and BARD terms, never replace stored fields.
    "cache_dir": str(ROOT / "analysis/exp3_numeric_cache"),
    "progress": True,  # Print each recording and throughput every 30 seconds.
    "minimum_per_stratum": 8,  # Explicit diagnostics for under-covered observed strata.
    "exp2_roots": [str(ROOT / "evaluation/exp2")],
    "exp2_loading": ["loaded"],  # Add unloaded for separate matched-control outputs.
    "sampling_seed": 0,
    "device": "cuda:0",
    "output_dir": str(ROOT / "analysis/exp3"),
    "dynamics": True,  # Direct BARD; requires CUDA, bard-pytorch-dynamics and Pinocchio.
    "urdf": None,  # Optional local relocation; must match recorded URDF hash.
    "fallback_predictions": True,  # Fill missing decoder fields only; stored outputs take precedence.
    "latex_method_names": {},  # Optional overrides keyed by method ID; defaults resolved below.
}
if os.environ.get("EXP3_CONFIG"):
    CONFIG.update(json.loads(os.environ["EXP3_CONFIG"]))

CONFIG = analysis.resolve_methods(CONFIG)
display(pd.DataFrame.from_dict(CONFIG["methods"], orient="index"))
# Resolve paper labels/macros even when methods were supplied by JSON overrides.
for method in CONFIG["methods"].values():
    if method["task"] in DISPLAY_NAMES:
        method["display_name"] = DISPLAY_NAMES[method["task"]]
CONFIG["latex_method_names"] = {
    **{key: LATEX_METHOD_NAMES[method["task"]] for key, method in CONFIG["methods"].items()
       if method["task"] in LATEX_METHOD_NAMES},
    **CONFIG.get("latex_method_names", {}),
}


,task,display_name
b1z1_pact,b1z1_pact,PILOT
b1z1_pact_ab10_actor,b1z1_pact_ab10_actor,A7
b1z1_pact_ab4_coupled_none,b1z1_pact_ab4_coupled_none,A4
b1z1_pact_ab5_coupled_concat,b1z1_pact_ab5_coupled_concat,A5
b1z1_pact_ab6_position_film,b1z1_pact_ab6_position_film,A1
b1z1_pact_ab7_position_full,b1z1_pact_ab7_position_full,A2
b1z1_pact_ab8_coupled_film,b1z1_pact_ab8_coupled_film,A3
b1z1_pact_ab9_representation,b1z1_pact_ab9_representation,A6
b1z1_ppo_pos,b1z1_ppo_pos,Pos
b1z1_unifp_original,b1z1_unifp_original,UniFP


## Execute and audit
Each model consumes only its own checkpoint's recorded history, including native Pos torque-history and UniFP compensation commands. No compatibility grouping, invented FF channels, substituted actor actions, or cross-policy histories are used. Selection identities stream to JSONL; aggregate manifests record available/selected counts.

Current force heads are scored against the **previous applied interval load visible in history_t**. Newly imposed loads are labelled separately. GRFs use the recorded source torque-conditioning input and successor-yaw conversion, and are compared with raw interval-average world forces, not processed training labels.

In [ ]:
result = analysis.run_analysis(CONFIG)
display(pd.Series({k:v for k,v in result['timings'].items() if k != 'runs'}, name='Runtime / cache counts'))
for path in sorted(Path(result["output_dir"]).glob("native_*_table.csv")):
    display(pd.read_csv(path, header=[0, 1], index_col=0))
print("\n".join(result["diagnostics"]) or "No coverage diagnostics.")
display(pd.DataFrame(result["unavailable"]))

Exp3: b1z1_pact_ab10_actor: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab10_actor/combine_in/seed_0


Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='floating_base']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='ee_gripper']


  16,384 transitions; 538.9/s; 0.5 min
  33,792 transitions; 554.7/s; 1.0 min
  51,200 transitions; 560.1/s; 1.5 min
  68,608 transitions; 563.4/s; 2.0 min
Exp3: b1z1_pact_ab10_actor: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab10_actor/combine_in/seed_1
  17,408 transitions; 553.1/s; 0.5 min
  34,816 transitions; 560.1/s; 1.0 min
  52,224 transitions; 562.3/s; 1.5 min
  69,632 transitions; 561.4/s; 2.1 min
Exp3: b1z1_pact_ab10_actor: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab10_actor/combine_in/seed_2
  16,384 transitions; 540.3/s; 0.5 min
  33,792 transitions; 546.6/s; 1.0 min
  51,200 transitions; 547.5/s; 1.6 min
  68,608 transitions; 552.3/s; 2.1 min
Exp3: b1z1_pact_ab10_actor: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab10_actor/ee_only_in/seed_0
  16,384 transitions; 535.2/s; 0.5 min
  33,792 transit

Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='floating_base']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='ee_gripper']


  17,408 transitions; 577.8/s; 0.5 min
  35,840 transitions; 585.6/s; 1.0 min
  54,272 transitions; 587.6/s; 1.5 min
  72,704 transitions; 589.1/s; 2.1 min
Exp3: b1z1_pact_ab4_coupled_none: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none/combine_in/seed_1
  17,408 transitions; 570.2/s; 0.5 min
  35,840 transitions; 577.3/s; 1.0 min
  54,272 transitions; 582.0/s; 1.6 min
  72,704 transitions; 582.0/s; 2.1 min
Exp3: b1z1_pact_ab4_coupled_none: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none/combine_in/seed_2
  17,408 transitions; 567.5/s; 0.5 min
  35,840 transitions; 582.7/s; 1.0 min
  54,272 transitions; 584.6/s; 1.5 min
  72,704 transitions; 587.8/s; 2.1 min
Exp3: b1z1_pact_ab4_coupled_none: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab4_coupled_none/ee_only_in/seed_0
  18,432 transition

...

  18,432 transitions; 587.9/s; 0.5 min
  36,864 transitions; 590.8/s; 1.0 min
  55,296 transitions; 591.5/s; 1.6 min
  73,728 transitions; 593.5/s; 2.1 min
Exp3: b1z1_pact_ab5_coupled_concat: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab5_coupled_concat/combine_in/seed_1
  17,408 transitions; 574.7/s; 0.5 min
  34,816 transitions; 576.4/s; 1.0 min
  53,248 transitions; 582.0/s; 1.5 min
  71,680 transitions; 584.1/s; 2.0 min
Exp3: b1z1_pact_ab5_coupled_concat: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab5_coupled_concat/combine_in/seed_2
  17,408 transitions; 578.9/s; 0.5 min
  35,840 transitions; 588.0/s; 1.0 min
  54,272 transitions; 590.0/s; 1.5 min
  72,704 transitions; 592.2/s; 2.0 min
Exp3: b1z1_pact_ab5_coupled_concat: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab5_coupled_concat/ee_only_in/seed_0
  18,43

Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='floating_base']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='ee_gripper']


  18,432 transitions; 587.8/s; 0.5 min
  36,864 transitions; 599.6/s; 1.0 min
  56,320 transitions; 604.8/s; 1.6 min
Exp3: b1z1_pact_ab6_position_film: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab6_position_film/combine_in/seed_1
  18,432 transitions; 599.2/s; 0.5 min
  36,864 transitions; 603.4/s; 1.0 min
  55,296 transitions; 605.7/s; 1.5 min
  73,728 transitions; 606.9/s; 2.0 min
Exp3: b1z1_pact_ab6_position_film: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab6_position_film/combine_in/seed_2
  18,432 transitions; 597.2/s; 0.5 min
  36,864 transitions; 604.7/s; 1.0 min
  55,296 transitions; 606.3/s; 1.5 min
  73,728 transitions; 606.1/s; 2.0 min
Exp3: b1z1_pact_ab6_position_film: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab6_position_film/ee_only_in/seed_0
  18,432 transitions; 601.3/s; 0.5 min
  36,864 tran

Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='floating_base']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='FL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RR_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='RL_foot_fixed']
Unknown attribute "dont_collapse" in /robot[@name='b1z1_description']/joint[@name='ee_gripper']


  18,432 transitions; 600.9/s; 0.5 min
  36,864 transitions; 605.6/s; 1.0 min
  55,296 transitions; 606.1/s; 1.5 min
  74,539 transitions; 609.1/s; 2.0 min
Exp3: b1z1_pact_ab7_position_full: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab7_position_full/combine_in/seed_1
  18,432 transitions; 604.7/s; 0.5 min
  36,864 transitions; 608.7/s; 1.0 min
  56,320 transitions; 611.9/s; 1.5 min
Exp3: b1z1_pact_ab7_position_full: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab7_position_full/combine_in/seed_2
  18,432 transitions; 597.2/s; 0.5 min
  36,864 transitions; 603.4/s; 1.0 min
  55,296 transitions; 605.0/s; 1.5 min
  73,728 transitions; 606.6/s; 2.0 min
Exp3: b1z1_pact_ab7_position_full: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/evaluation/exp1/b1z1_pact_ab7_position_full/ee_only_in/seed_0
  18,432 transitions; 594.1/s; 0.5 min
  36,864 tran

## Portable results
The page-wide tables contain EE force, torso force, torso moment, GRF, base-linear velocity, base-angular velocity, and joint-velocity component RMSE. Biases, individual axes/feet, arm/leg velocities, event phases and motion modes remain in CSV/JSON breakdowns. Table means first reduce within trials, then equally across evaluation seeds; SD uses ddof=1. Missing heads/dynamics are dashes, never zeros.

BARD uses q_t mechanics, actual interval-average executed torque, and frozen decoded loads for an analytic one-step velocity prediction. Measured-loading and constant-velocity references are separate variants. Native UniFP has no complete learned-loading dynamics result because it has no moment or GRF head. Physics losses being disabled in training does not disable these offline diagnostics.

In [ ]:
display(pd.DataFrame(result["per_seed"]).head(20))
display(pd.DataFrame(result["native_parity"].items(), columns=["Recording", "Native parity"]))
display(pd.DataFrame([r for r in result["table_summary"] if r["variant"] != 'decoded']))
display(pd.DataFrame(result["dynamics_approximations"]).head())
print("Manifest, native/reference diagnostics and native LaTeX tables:", result["output_dir"])
print("Selection:", result["selection_mode"], "eligible:", result["selection"]["available"], "selected:", result["selection"]["selected"])

## Action channels and whole-body support (Exp2)
Stream loaded pushing/pulling/carrying recordings for PILOT, UniFP, Pos, A1, A3, A6. Missing recordings/fields are reported, never collected. Dashed event lines use recorded timing. Loads/channels/support use interval timestamps; tracking uses successors. Bands show checkpoint SD when independently trained checkpoints exist, otherwise evaluation-seed SD; surviving trial counts are plotted.

Geometry is URDF FK of the **mean delayed position command** at measured base_t versus reference_t, an explicit approximation when substep targets vary. Torque RMS uses recorded substep squares; PD/FF are weighted motor-scaled requests, executed torques are clipped. Recorded CoM moments are averaged cross-products, not cross-products of means. Non-foot resultant contacts are included where recorded. Neither channel usage nor support moments uniquely identify compensation.

Offline dynamics include rotor armature and zero-target passive viscous damping. Unknown constraint-dependent friction torque and unrecorded stiffness-drive targets are omitted and labelled as approximations, not silently treated as measured zeros. Measured-loading versus decoded-loading errors help distinguish model mismatch from representation error. Support settling/overshoot are unavailable without a justified stationary reference.

In [ ]:
import exp3_support
support = exp3_support.analyze(CONFIG)
display(pd.DataFrame(support["summary"]))
display(pd.DataFrame(support["comparisons"]))
print("\n".join(support["diagnostics"]))
from IPython.display import Image
for path in sorted((Path(CONFIG["output_dir"]) / "action_support").glob("*.png")):
    display(Image(filename=str(path)))